# 赤い箱をつかんで右へ運ぶ — Genesis × 模倣学習

**Docker / JupyterLab / CPUで実行する入門版。** 上から順にセルを実行します（Shift + Enter）。

- **学習**：公開されたロボットの実演を使い、小さなニューラルネットに動作を覚えさせます。
- **推論**：覚えた重みから腕の目標位置と指の力を計算し、Genesisの物理演算でロボットを動かします。
- **入力**：箱の初期XY座標、シミュレーターで既知の重さ、時刻。**画像・赤色認識や言語理解は行いません。**
- **出力**：手の目標XYZ座標と把持力。関節への変換は逆運動学、追従はPD制御です。

「右」は固定カメラから見た右（世界座標の−X方向）。赤い箱を青い台に置きます。
モデルは時系列の軌道を生成する方式なので、途中で箱を動かした場合の再計画・つかみ直しはできません。

初回のGenesisコンパイルには数分以上かかることがあります。処理中はセル左に `[*]` が出ます。

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "robot_lab").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "robot_lab").is_dir(), "プロジェクト内から開いてください"
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("EGL_PLATFORM", "surfaceless")
os.environ.setdefault("LIBGL_ALWAYS_SOFTWARE", "1")
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Video, Markdown, Image
from io import BytesIO
def show_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close(fig)
from robot_lab.dataset import load_dataset, split_episodes
from robot_lab.policy import train, load_policy, evaluate_offline
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("作業フォルダ:", ROOT)

## 1. 既存の学習データを取得・確認

出典：[jalal06/tactile-genesis-heavy-mg-v2](https://huggingface.co/datasets/jalal06/tactile-genesis-heavy-mg-v2)。初回は公開実演24件を固定revisionからインターネット経由で取得し、数値データだけを保存します。取得には時間・通信量・ディスク容量が必要です。2回目以降は保存済みデータを使用します。

元データの灰色の箱を本環境では赤色で表示します。床・Panda・箱・配置台の簡略環境であり、元の障害物やバスケットを含む環境の完全再現ではありません。`state[t]` の次の指令は `action[t+1]` です。

In [ ]:
from robot_lab.dataset import download_dataset
if not Path("data/compact/manifest.json").exists():
    print("公開データ24件を初回取得します。ネット接続が必要です。", flush=True)
    download_dataset()
episodes, manifest = load_dataset()
splits = split_episodes(episodes, seed=42)
print("出典:", manifest["repo"])
print("固定revision:", manifest["revision"])
for split, group in splits.items():
    print(split, len(group), [e["id"] for e in group])
print("一実演:", episodes[0]["action"].shape, "数値データのみ")

実演単位で **学習16件・検証3件・テスト5件** に分けます。検証データで学習時点を選び、独立したテストデータで評価します。

In [ ]:
demo = splits["train"][0]
t = np.arange(1,158)/60
fig, axes = plt.subplots(1,2,figsize=(11,3.3))
for j, label in enumerate(["x", "y", "z"]):
    axes[0].plot(t, demo["action"][1:,j], label=label)
axes[0].set(xlabel="Time (s)",ylabel="Hand target (m)",title="Public expert commands")
axes[0].legend()
axes[1].plot(t, demo["action"][1:,7])
axes[1].set(xlabel="Time (s)",ylabel="Grip force / 45 N",title=demo["id"])
fig.tight_layout(); show_figure(fig)

## 2. ニューラルネットを学習

5件を独立してテストするため、初回は必ず再学習します。標準450エポック、CPUで実行します。


In [ ]:
RETRAIN = True
EPOCHS = 450
if RETRAIN:
    model, metadata, history = train(splits, epochs=EPOCHS, device="cpu")
else:
    model, metadata = load_policy()
    if metadata["test_ids"] != [e["id"] for e in splits["test"]]:
        raise ValueError("5件テスト用のモデルではありません。RETRAIN=Trueで学習してください。")
    history = json.loads(Path("outputs/model/history.json").read_text())
print("検証データで選ばれたepoch:", metadata["best_epoch"])


In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot([x["epoch"] for x in history],[x["train_loss"] for x in history],label="train")
ax.plot([x["epoch"] for x in history],[x["validation_loss"] for x in history],label="validation")
ax.set(xlabel="Epoch",ylabel="Loss",yscale="log",title="Behavior cloning")
ax.legend(); show_figure(fig)
metrics = evaluate_offline(model, splits["test"])
Path("outputs/offline_metrics.json").write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))

`position_rmse_m` は指令位置の予測誤差です。**小さくても、つかむことに成功した証明にはなりません。**
次に物理シミュレーションで箱の持ち上げ・運搬・解放・静止を確かめます。

## 3. Genesisで推論を実行

Genesisは別プロセスで実行します。Jupyterのカーネルに初期化状態を残さず、繰り返し実行できます。
テスト5件の最初を使用します。学習した位置と把持力を実際のモーターに渡します。
箱を手に固定する処理や、箱の位置を直接書き換える処理はありません。

初回はコンパイルが長めです。下のセルは待機中もログを表示します。

In [ ]:
def run_sim(episode, mode="policy", video=True):
    out = ROOT / "outputs" / f"{mode}_{episode}"
    cmd = [sys.executable, "-m", "robot_lab.simulation", "--mode", mode,
           "--episode", episode, "--output", str(out)]
    if not video: cmd.append("--no-video")
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    if code:
        raise RuntimeError(f"Genesis失敗: exit={code}。上のログを確認してください。")
    return json.loads((out / "result.json").read_text()), out

TEST_ID = splits["test"][0]["id"]
result, out = run_sim(TEST_ID)
print("物理タスク成功:", result["success"])

In [ ]:
display(Video(str(out / "rollout.mp4"), embed=True, width=640))
print("最大高さ:", round(result["maximum_cube_height_m"],3), "m")
print("配置誤差:", round(result["goal_distance_m"],3), "m")
print({k:result[k] for k in ["lifted","placed","supported","stable","released"]})

判定条件：高さ10 cmを超えて持ち上がる／青い台の中心から4.5 cm以内／台上の高さで静止／最後の把持指令が開く状態。
最後の0.5秒間に5 mm以上動いた場合も失敗です。各条件と座標はJSON・CSVに保存します。

画面に表示する実演は約3.6秒です。下の任意セルで5件すべてを検証できます。少数件の成功率は広い条件への汎化性能を保証しません。

In [ ]:
RUN_ALL_TESTS = True  # Trueでテスト5件すべてを実行（CPUでは時間がかかります）
if RUN_ALL_TESTS:
    results = []
    for episode in splits["test"]:
        item, _ = run_sim(episode["id"])
        results.append(item)
    print("成功:", sum(x["success"] for x in results), "/", len(results))
    Path("outputs/physics_evaluation.json").write_text(json.dumps(results,indent=2))

## 4. うまくつかめなかったときの切り分け（任意）

`replay` は公開データの正解指令を再生する診断モードです。**学習モデルの推論結果として発表しないでください。**
これも失敗する場合は、環境や制御設定の差が疑われます。
本体の環境定数を調整する場合は学習・検証側の実演を使い、テスト側を繰り返し調整に使わないでください。

In [ ]:
RUN_EXPERT_REPLAY = False
if RUN_EXPERT_REPLAY:
    replay_result, replay_out = run_sim(splits["train"][0]["id"], mode="replay")
    display(Video(str(replay_out / "rollout.mp4"),embed=True,width=640))

## 5. データの再取得（任意）

取得済みデータは通常そのまま利用できます。再取得する場合は次のセルを実行してください。既に `data/raw` にある元ファイルは再利用します。

In [ ]:
REDOWNLOAD = False
if REDOWNLOAD:
    from robot_lab.dataset import download_dataset
    download_dataset()

## 6. 動作映像だけの提出動画を作る

5件の推論映像を0.25倍速でつなぎ、約72秒の動画を作ります。字幕・説明画面・音声はありません。初回は上のセルで学習し、テスト5件を全て動画付きで実行してください。再学習時も5件を撮り直します。

In [ ]:
EXPORT_PRESENTATION = True
if EXPORT_PRESENTATION:
    subprocess.run([sys.executable, "-m", "robot_lab.export_video"], check=True, cwd=ROOT)
    display(Video("outputs/submission_80s.mp4", embed=True, width=720))

## 発表時に説明すること

「公開データで学習したMLPが、箱の初期位置・既知の重さ・時刻から手先の目標と把持力を推論する。
Genesisの物理演算でPandaが赤い箱をつかみ、画面右の青い台に運ぶ」と説明できます。

元データは灰色の箱・バスケット・障害物を含みます。本プログラムはその動作指令を学び、簡略化した環境で実行しています。
赤色を見分けるAI、言葉の意味を理解するAI、失敗から自力で立て直すAIではありません。

提出動画は1〜3分の要件に合わせ、目的・データ・学習結果・推論映像・限界を説明してください。
短い `rollout.mp4` だけでは1分に足りません。自動生成する約72秒版を利用できます。動画には説明を重ねていないため、課題の説明は別途レポートで補います。